# DESC resolution and the vacuum equilibrium

What resolution must an equilibrium dataset be solved at? Two curves over one `(L, M, N)` grid,
the first after the truncation of Merlo et al. (2021), section 2.2.2, the second this work's:

- **truncated**: the reference solution cut to a smaller resolution. What that resolution can
  *represent*.
- **re-solved**: the truncation solved again at that resolution. What a dataset generated there
  would *carry*.

Their gap is the part of the truncation error a solver recovers. `run_desc_resolution_analysis.py`
writes `mhd_resolution_force_residual.csv` (command in its docstring); this notebook only
summarizes and plots it. The last section reads the vacuum equilibrium of the shared
boundary, whose flux surfaces sit beside the scan in the thesis figure.

In [ ]:
import pathlib

import desc.grid
import desc.magnetic_fields
import desc.objectives
import desc.optimize
import desc.plotting
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import ticker
from plasma_state_estimation_for_diagnostic_placement_optimization.ideal_mhd import (
    desc_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.ideal_mhd import (
    plot as ideal_mhd_plot,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.mhd_equilibrium_forward_models import (  # noqa: E501
    dataset_utils,
    evaluation_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    pubfig,
)

# Anchored to the package, not the kernel's working directory, so the notebook runs from
# anywhere.
ROOT = pathlib.Path(pubfig.__file__).parents[2]
NOTEBOOK_DIR = (
    ROOT
    / "results_plots_tables"
    / "results"
    / "mhd_equilibrium_forward_models"
    / "desc_resolution_analysis"
)
# Written by run_desc_resolution_analysis.py, not by this notebook.
SCAN_CSV = NOTEBOOK_DIR / "mhd_resolution_force_residual.csv"
PLOTS_TABLES_DIR = NOTEBOOK_DIR / "plots_tables"
PLOTS_TABLES_DIR.mkdir(exist_ok=True)
assert SCAN_CSV.exists(), (
    f"{SCAN_CSV.name} is missing; run run_desc_resolution_analysis.py first"
)
# The residual a generated dataset is required to reach, drawn on both figures.
RESOLUTION_THRESHOLD = 1.0e-2
# `(L, M, N)` the reference equilibria of `equilibrium_a_3k` were solved at.
REFERENCE_RESOLUTION = (11, 11, 12)
# Mathtext rather than a literal glyph, to stay ASCII.
FORCE_RESIDUAL_LABEL = (
    r"$\langle |\mathbf{F}| \rangle_\mathrm{vol}\, /\, "
    r"\langle |\nabla (B^2 / 2\mu_0)| \rangle_\mathrm{vol}$"
)

pubfig.apply_style()
scan = (
    pd.read_csv(SCAN_CSV)
    .sort_values(["n_coefficients", "equilibrium_id"])
    .reset_index(drop=True)
)
print(
    f"{len(scan)} rows, {scan.equilibrium_id.nunique()} equilibria, "
    f"{scan.n_coefficients.nunique()} resolutions, "
    f"reference at {int(scan.reference_n_coefficients.iloc[0])} coefficients"
)
scan.head()

One row per resolution. The statistics are geometric because the residual spans decades and is
heavy tailed: the geometric mean, and the standard deviation of `log10` as a multiplicative
spread (0.3 is about a factor of two either way).

In [ ]:
residual_columns = ("fnorm_truncated", "fnorm_resolved", "fnorm_reference")
prepared = scan.copy()
for column in residual_columns:
    positive = prepared[column].where(prepared[column] > 0.0)
    prepared[f"{column}_log"] = np.log10(positive)
aggregations = {
    "n_coefficients": ("n_coefficients", "first"),
    "n_equilibria": ("fnorm_resolved", "size"),
    "solve_seconds": ("solve_seconds", "mean"),
}
for column in residual_columns:
    aggregations[f"{column}_log_mean"] = (f"{column}_log", "mean")
    aggregations[f"{column}_log_std"] = (f"{column}_log", "std")
summary = prepared.groupby(
    ["radial_resolution", "poloidal_resolution", "toroidal_resolution"],
    as_index=False,
).agg(**aggregations)
for column in residual_columns:
    summary[f"{column}_geometric_mean"] = 10.0 ** summary[f"{column}_log_mean"]
summary = summary.sort_values("n_coefficients").reset_index(drop=True)
summary.to_csv(PLOTS_TABLES_DIR / "mhd_resolution_summary.csv", index=False)
columns = [
    "radial_resolution",
    "poloidal_resolution",
    "toroidal_resolution",
    "n_coefficients",
    "solve_seconds",
    "fnorm_truncated_geometric_mean",
    "fnorm_truncated_log_std",
    "fnorm_resolved_geometric_mean",
    "fnorm_resolved_log_std",
]
summary[columns]

Panel (b) of the thesis figure. Points are geometric means and bars one log-space standard deviation, against the coefficient
count, since no single mode number orders `(L, M, N)`. Red dashed: the criterion at `1e-2`.
Dotted: the equilibria's own residual at the full `(11, 11, 12)`, which neither curve passes.

In [ ]:
# The height of the vacuum flux surfaces below, which sit beside it in the thesis.
figure, axes = pubfig.subplots(width=3.2, aspect=0.93)
panel = axes[0]
for column, label, marker in (
    ("fnorm_truncated", "truncated", "o"),
    ("fnorm_resolved", "re-solved", "s"),
):
    center = summary[f"{column}_geometric_mean"]
    spread = summary[f"{column}_log_std"]
    panel.errorbar(
        summary["n_coefficients"],
        center,
        yerr=[
            center - center * 10.0**-spread,
            center * 10.0**spread - center,
        ],
        marker=marker,
        markersize=3.5,
        linewidth=1.2,
        capsize=2.0,
        label=label,
    )
# The criterion, deliberately unlabelled: it is a line to read the curves against
# rather than a third series, and naming it in the legend invites reading it as one.
panel.axhline(RESOLUTION_THRESHOLD, color="tab:red", linestyle="--", linewidth=1.0)
panel.axhline(
    float(summary["fnorm_reference_geometric_mean"].iloc[0]),
    color="0.45",
    linestyle=":",
    linewidth=1.0,
    label=f"full resolution {REFERENCE_RESOLUTION}",
)
panel.set_xscale("log")
panel.set_yscale("log")
# The default log locator thins the decade labels and places no minor ticks.
panel.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=15))
panel.yaxis.set_minor_locator(
    ticker.LogLocator(base=10.0, subs=(2, 3, 4, 5, 6, 7, 8, 9), numticks=15)
)
panel.yaxis.set_minor_formatter(ticker.NullFormatter())
# Less than a decade of coefficients gets a single default log label, so label round
# values that double each step and sit evenly on the log axis.
coefficient_ticks = (250, 500, 1000, 2000)
panel.set_xticks(coefficient_ticks)
panel.set_xticklabels([str(value) for value in coefficient_ticks])
panel.xaxis.set_minor_formatter(ticker.NullFormatter())
# Wide enough to hold every labelled tick: the data span 188 to 1565 coefficients, so
# on the automatic limits the 250 and 2000 ticks fall outside and are dropped without
# a warning.
panel.set_xlim(0.85 * float(summary["n_coefficients"].min()), 2200.0)
panel.set_xlabel("n spectral coefficients")
panel.set_ylabel(FORCE_RESIDUAL_LABEL)
# Room below the re-solved curve for the legend, which the full-resolution line
# shows through.
panel.set_ylim(bottom=2.0e-4)
panel.legend(
    loc="lower left",
    frameon=True,
    framealpha=0.7,
    edgecolor="none",
    handletextpad=0.4,
)
pubfig.save(figure, str(PLOTS_TABLES_DIR / "mhd_resolution_force_residual"))

The same curves per `(L, M, N)` triple, grouped by `L`: read it for which axis to spend
resolution on.

In [ ]:
summary_by_axis = summary.sort_values(
    ["radial_resolution", "poloidal_resolution"], ascending=[True, True]
)
labels = [
    rf"${row.radial_resolution} \times {row.poloidal_resolution}"
    rf" \times {row.toroidal_resolution}$"
    for row in summary_by_axis.itertuples()
]
positions = np.arange(len(summary_by_axis))
radial = summary_by_axis["radial_resolution"].to_numpy()
# Where each run of equal radial resolution starts, after the first.
block_starts = [
    index for index in range(1, len(radial)) if radial[index] != radial[index - 1]
]
block_edges = [0, *block_starts, len(radial)]

figure, axes = pubfig.subplots(width="double", aspect=0.42)
panel = axes[0]
for order, (column, label, marker) in enumerate(
    (("fnorm_truncated", "truncated", "o"), ("fnorm_resolved", "re-solved", "s"))
):
    center = summary_by_axis[f"{column}_geometric_mean"].to_numpy()
    spread = summary_by_axis[f"{column}_log_std"].to_numpy()
    color = pubfig.PALETTE[order % len(pubfig.PALETTE)]
    # One call per radial block, so the connecting line stops at each boundary.
    # Drawn across them it would read as a trend where it is a reset: the residual
    # jumps back up from the finest poloidal case of one block to the coarsest
    # of the next, and a line through that says the opposite of what the data does.
    for first, last in zip(block_edges[:-1], block_edges[1:]):
        inside = slice(first, last)
        panel.errorbar(
            positions[inside],
            center[inside],
            yerr=[
                center[inside] - center[inside] * 10.0 ** -spread[inside],
                center[inside] * 10.0 ** spread[inside] - center[inside],
            ],
            marker=marker,
            markersize=3.5,
            linewidth=1.2,
            capsize=2.0,
            color=color,
            label=label if first == 0 else None,
        )
# The criterion, deliberately unlabelled: it is a line to read the curves against
# rather than a third series, and naming it in the legend invites reading it as one.
panel.axhline(RESOLUTION_THRESHOLD, color="tab:red", linestyle="--", linewidth=1.0)
panel.axhline(
    float(summary_by_axis["fnorm_reference_geometric_mean"].iloc[0]),
    color="0.45",
    linestyle=":",
    linewidth=1.0,
    label=f"full resolution {REFERENCE_RESOLUTION}",
)
for start in block_starts:
    panel.axvline(start - 0.5, color="0.85", linewidth=0.6, zorder=0)

panel.set_xticks(positions)
# Vertical, not the usual 45 degrees: the labels contain a times sign, and a glyph
# with four-fold symmetry rotated by 45 degrees lands looking exactly like a plus,
# so "9 x 9 x 9" reads as a sum. At 90 degrees the times sign maps onto itself.
panel.set_xticklabels(labels, rotation=90.0, ha="center")
panel.set_xlim(-0.6, len(summary_by_axis) - 0.4)
panel.set_yscale("log")
# The default log locator thins the decade labels and places no minor ticks.
panel.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=15))
panel.yaxis.set_minor_locator(
    ticker.LogLocator(base=10.0, subs=(2, 3, 4, 5, 6, 7, 8, 9), numticks=15)
)
panel.yaxis.set_minor_formatter(ticker.NullFormatter())
panel.set_xlabel(r"$L \times M \times N$")
panel.set_ylabel(FORCE_RESIDUAL_LABEL)
panel.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    ncol=1,
    frameon=False,
    handletextpad=0.4,
    borderaxespad=0.0,
)
pubfig.save(figure, str(PLOTS_TABLES_DIR / "mhd_resolution_by_axis"))

## Vacuum equilibrium

The `p0 = 0` equilibrium on the fixed boundary every dataset of this area shares, row `DMaPpnz4PaGM4qMvDw2E8Ak` of the interpolator parquet, at the dataset's unit scale (`R0 ~ 1 m`, `B ~ 1 T`).

In [ ]:
DATA_DIRECTORY = ROOT.parents[1] / "data"
DATASET = "equilibrium_a_interpolator_28_spline_input"
VACUUM_EQUILIBRIUM_ID = "DMaPpnz4PaGM4qMvDw2E8Ak"

metadata = dataset_utils.load_metadata(DATA_DIRECTORY / f"{DATASET}_settings.json")
dataset = dataset_utils.load_dataset(DATA_DIRECTORY / f"{DATASET}.parquet", metadata)
row = dataset.equilibrium_ids.index(VACUUM_EQUILIBRIUM_ID)
(stored_equilibrium,) = dataset_utils.target_equilibria(
    dataset_utils.slice_dataset(dataset, [row])
)
eq = desc_utils.desc_equilibrium_to_desc_object(
    stored_equilibrium, metadata.basis.resolution
)
print(
    f"<beta>_vol = {dataset.beta_vol[row]:g}, "
    f"fnorm = {evaluation_utils.force_residual(eq):.2e}, "
    f"NFP = {eq.NFP}, Psi = {eq.Psi:.4f} Wb"
)

Panel (a) of the thesis figure. Stellarator symmetry maps the cross section at `phi` onto the one at `2 pi / NFP - phi` mirrored in `Z`, so four angles over half a field period, `0` to `pi / NFP`, show every distinct shape.

In [ ]:
phi = np.linspace(0.0, np.pi / eq.NFP, 4)
# One scale for every panel, so the shapes compare directly; each row only spans
# its own `Z` range, which keeps the figure the height of the resolution scan.
major_radius_limits = (0.73, 1.19)
height_limits = ((-0.32, 0.39), (-0.13, 0.30))
figure, axes = pubfig.subplots(
    2,
    2,
    width=2.4,
    aspect=1.25,
    sharex=True,
    sharey="row",
    gridspec_kw={"height_ratios": [top - bottom for bottom, top in height_limits]},
)
ideal_mhd_plot.plot_surfaces(
    stored_equilibrium,
    rho=np.linspace(0.0, 1.0, 7)[1:],
    theta=np.array([]),
    phi=phi,
    ax=axes,
    rho_color=pubfig.PALETTE[0],
    rho_lw=0.7,
    lcfs_lw=1.3,
    axis_size=5,
)
for panel, title in zip(
    axes.flat,
    (r"$\phi = 0$", r"$\phi = \pi/9$", r"$\phi = 2\pi/9$", r"$\phi = \pi/3$"),
    strict=True,
):
    panel.set_title(title)
    panel.xaxis.set_major_locator(ticker.MultipleLocator(0.2))
    panel.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    panel.set_xlim(*major_radius_limits)
    panel.label_outer()
for row_axes, limits in zip(axes, height_limits, strict=True):
    row_axes[0].set_ylim(*limits)
pubfig.save(figure, str(PLOTS_TABLES_DIR / "mhd_vacuum_flux_surfaces"))

Quasi-isodynamicity (QI) residual from DESC's `Omnigenity` (Dudt et al. 2024, doi:10.1017/S0022377824000151): on each surface an omnigenous field with helicity `(0, NFP)` and a straight `B_max` contour (`FixOmniBmax`) is fitted to the Boozer `|B|`, the equilibrium fixed. The residual, this work's metric, is the RMS `|B|` misfit relative to the surface mean and depends on the fitted field's resolution. DESC assumes `B_max` at `theta = zeta = 0`, checked first. About 4 min.

In [ ]:
SURFACES_RHO = (0.25, 0.5, 0.75, 1.0)
N_WELL_KNOTS = 6
"""Spline knots of the fitted well |B|(eta), DESC's `M_B`."""
N_MAP_MODES = 2
"""Fourier modes in eta and alpha of the fitted map, DESC's `M_x` and `N_x`."""

check_grid = desc.grid.LinearGrid(
    rho=np.array([1.0]),
    theta=np.array([0.0]),
    zeta=np.array([0.0, np.pi / eq.NFP]),
    NFP=eq.NFP,
)
field_strength_at_check = np.asarray(eq.compute("|B|", grid=check_grid)["|B|"])
assert field_strength_at_check[0] > field_strength_at_check[1], (
    "B_max is not at zeta = 0"
)

residual_rows = []
for rho in SURFACES_RHO:
    surface_grid = desc.grid.LinearGrid(
        rho=np.array([rho]), M=4 * eq.M, N=4 * eq.N, NFP=eq.NFP, sym=False
    )
    field_strength = np.asarray(eq.compute("|B|", grid=surface_grid)["|B|"])
    field = desc.magnetic_fields.OmnigenousField(
        L_B=0,
        M_B=N_WELL_KNOTS,
        L_x=0,
        M_x=N_MAP_MODES,
        N_x=N_MAP_MODES,
        NFP=eq.NFP,
        helicity=(0, eq.NFP),
        B_lm=np.linspace(field_strength.min(), field_strength.max(), N_WELL_KNOTS),
    )
    field_grid = desc.grid.LinearGrid(
        rho=np.array([rho]), theta=2 * N_WELL_KNOTS, N=eq.N, NFP=eq.NFP, sym=False
    )
    omnigenity = desc.objectives.Omnigenity(
        eq=eq,
        field=field,
        eq_fixed=True,
        eq_grid=surface_grid,
        field_grid=field_grid,
    )
    objective = desc.objectives.ObjectiveFunction(omnigenity)
    objective.build(verbose=0)
    (field,), _ = desc.optimize.Optimizer("lsq-exact").optimize(
        field,
        objective,
        constraints=(desc.objectives.FixOmniBmax(field=field),),
        maxiter=100,
        verbose=0,
        copy=True,
    )
    error = np.asarray(omnigenity.compute(field.params_dict))
    rms_error = float(np.sqrt(np.mean(error**2)))
    residual_rows.append(
        {
            "rho": rho,
            "rms_error_T": rms_error,
            "relative_rms_error": rms_error / float(field_strength.mean()),
        }
    )
qi_residual = pd.DataFrame(residual_rows)
qi_residual.to_csv(PLOTS_TABLES_DIR / "mhd_vacuum_qi_residual.csv", index=False)
print(qi_residual.to_string(index=False))

`|B|` in Boozer coordinates on two surfaces, with four field lines. In a QI field the contours close poloidally and the `B_max` contour is straight.

In [ ]:
# A plain figure, since DESC lays out its own colorbar.
figure, axes = plt.subplots(1, 2, figsize=(10.0, 4.0))
for axis, rho in zip(axes.flat, (0.5, 1.0), strict=True):
    desc.plotting.plot_boozer_surface(eq, rho=rho, fieldlines=4, ax=axis)
    axis.set_title(rf"$\rho = {rho:g}$")
pubfig.save(figure, str(PLOTS_TABLES_DIR / "mhd_vacuum_boozer_surface"))